# Explorador de KPIs por tipo de dado

Lê a `gold_daily` e a `gold_monthly` do HDFS e permite ver, por **tipo de dado**:
- **Diário**: escolher tipo + distrito + mês (mostra os dias desse distrito).
- **Mensal**: escolher tipo + mês (mostra os 18 distritos).

In [44]:
!pip install -q ipywidgets
import pandas as pd
from pyspark.sql import SparkSession
from ipywidgets import interact, Dropdown

import ipywidgets as widgets
from ipywidgets import interact
import numpy as np
import matplotlib.pyplot as plt


spark = SparkSession.builder.getOrCreate()

GOLD_DAILY   = "hdfs://hadoop-master:39000/2250055/gold/gold_daily"
GOLD_MONTHLY = "hdfs://hadoop-master:39000/2250055/gold/gold_monthly"

# Tabelas pequenas -> carregam-se uma vez para pandas (filtragem instantanea).
g_daily   = spark.read.parquet(GOLD_DAILY).toPandas()
g_monthly = spark.read.parquet(GOLD_MONTHLY).toPandas()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print("daily:", g_daily.shape, "| monthly:", g_monthly.shape)


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
daily: (9432, 31) | monthly: (324, 48)


In [45]:
# Grupos de colunas por tipo de dado
COLS_DAILY = {
 "Temperatura": ["temp_max","temp_min","temp_media","amplitude_termica"],
 "Humidade": ["humidade_media","humidade_max","humidade_min","amplitude_humidade"],
 "Vento": ["vento_media","vento_max","rajada_max","categoria_vento"],
 "UV": ["uv_index","categoria_uv"],
 "Sol/Luz": ["sunhour","duracao_dia","aproveitamento_solar"],
 "Precipitacao": ["precip_total"],
 "Sensacoes": ["heat_index","wind_chill"],
 "Flags": ["dia_quente","dia_frio","dia_neve","dia_chuva","dia_chuva_intensa"],
}
COLS_MONTHLY = {
 "Temperatura": ["temp_max_mes","temp_min_mes","temp_media_mes","amplitude_extrema_mes","amplitude_media_dia","temp_desvio","graus_dia_aquecimento","graus_dia_arrefecimento","maior_onda_calor"],
 "Humidade": ["humidade_media_mes","humidade_max_mes","humidade_min_mes","amplitude_humidade_media"],
 "Vento": ["vento_media_mes","vento_max_mes","rajada_max_mes","vento_cat_fraco","vento_cat_moderado","vento_cat_forte","vento_cat_muito_forte"],
 "UV": ["uv_max_mes","uv_media_mes"],
 "Sol/Luz": ["sunhour_total_mes","sunhour_media_mes","duracao_dia_media","aproveitamento_solar_media"],
 "Precipitacao": ["precip_total_mes","precip_max_dia_mes","maior_periodo_seco"],
 "Sensacoes": ["heat_index_max_mes","wind_chill_min_mes"],
 "Contagens (dias)": ["dias_quentes","dias_frios","dias_humidos","dias_secos","dias_vento_forte","dias_uv_alto","dias_uv_extremo","dias_chuva","dias_chuva_intensa","dias_neve"],
 "Comparativos": ["ranking_temp","anomalia_temp"],
}

def diario(tipo, distrito, mes):
    """Dados diarios de UM distrito num mes, so as colunas do tipo escolhido."""
    d = g_daily[(g_daily["district"] == distrito) & (g_daily["year_month"] == mes)]
    return d.sort_values("date")[["date"] + COLS_DAILY[tipo]].reset_index(drop=True)

def mensal(tipo, mes):
    """Dados mensais de TODOS os distritos num mes, so as colunas do tipo escolhido."""
    d = g_monthly[g_monthly["year_month"] == mes]
    return d.sort_values("district")[["district"] + COLS_MONTHLY[tipo]].reset_index(drop=True)

## Vista DIÁRIA — escolhe tipo, distrito e mês

In [46]:
distritos = sorted(g_daily["district"].unique())
meses_d   = sorted(g_daily["year_month"].unique())

interact(diario,
         tipo=Dropdown(options=list(COLS_DAILY), description="Tipo:"),
         distrito=Dropdown(options=distritos, description="Distrito:"),
         mes=Dropdown(options=meses_d, description="Mes:"));

interactive(children=(Dropdown(description='Tipo:', options=('Temperatura', 'Humidade', 'Vento', 'UV', 'Sol/Lu…

## Vista MENSAL — escolhe tipo e mês (mostra os 18 distritos)

In [53]:
meses_m = sorted(g_monthly["year_month"].unique())

interact(mensal,
         tipo=Dropdown(options=list(COLS_MONTHLY), description="Tipo:"),
         mes=Dropdown(options=meses_m, description="Mes:"));

interactive(children=(Dropdown(description='Tipo:', options=('Temperatura', 'Humidade', 'Vento', 'UV', 'Sol/Lu…

In [54]:
 ## Visualizar graficamente conjuntos de KPIs por tipo de dado

In [55]:
## Temperatura

In [56]:
GOLD_MONTHLY = "hdfs://hadoop-master:39000/2250055/gold/gold_monthly"

gm = (spark.read.parquet(GOLD_MONTHLY)
        .select("district", "year_month",
                "temp_max_mes", "temp_min_mes", "dias_quentes", "dias_frios")
        .toPandas())
gm["district"] = gm["district"].str.replace("%20", " ")
gm["ano"] = gm["year_month"].str.slice(0, 4).astype(int)
gm["mes"] = gm["year_month"].str.slice(5, 7).astype(int)

anos = sorted(gm["ano"].unique().tolist())
MESES = [("Janeiro", 1), ("Fevereiro", 2), ("Março", 3), ("Abril", 4),
         ("Maio", 5), ("Junho", 6), ("Julho", 7), ("Agosto", 8),
         ("Setembro", 9), ("Outubro", 10), ("Novembro", 11), ("Dezembro", 12)]
NOME_MES = {v: k for k, v in MESES}
METRICAS = ["Temperatura máxima", "Temperatura mínima", "Dias quentes vs frios"]

def mostrar_temperatura(ano, mes, metrica):
    d = gm[(gm["ano"] == ano) & (gm["mes"] == mes)].sort_values("district")
    if d.empty:
        print(f"Sem dados para {NOME_MES[mes]} {ano}.")
        return

    distritos = d["district"].tolist()
    x = np.arange(len(distritos))
    plt.figure(figsize=(13, 6))

    if metrica == "Temperatura máxima":
        plt.bar(x, d["temp_max_mes"], color="#d62728")
        plt.ylabel("Temperatura máxima do mês (°C)")
    elif metrica == "Temperatura mínima":
        plt.bar(x, d["temp_min_mes"], color="#1f77b4")
        plt.ylabel("Temperatura mínima do mês (°C)")
    else:
        w = 0.4
        plt.bar(x - w/2, d["dias_quentes"], width=w, label="dias quentes", color="#ff7f0e")
        plt.bar(x + w/2, d["dias_frios"],  width=w, label="dias frios",  color="#1f77b4")
        plt.ylabel("Número de dias")
        plt.legend()

    plt.xticks(x, distritos, rotation=45, ha="right")
    plt.title(f"{metrica} por distrito — {NOME_MES[mes]} {ano}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

interact(mostrar_temperatura,
         ano=widgets.Dropdown(options=anos, description="Ano:"),
         mes=widgets.Dropdown(options=MESES, value=1, description="Mês:"),
         metrica=widgets.Dropdown(options=METRICAS, description="Métrica:"));

interactive(children=(Dropdown(description='Ano:', options=(2025, 2026), value=2025), Dropdown(description='Mê…

In [57]:
## Humidade

In [58]:
gh = (spark.read.parquet(GOLD_MONTHLY)
        .select("district", "year_month", "humidade_max_mes", "humidade_min_mes")
        .toPandas())
gh["district"] = gh["district"].str.replace("%20", " ")
gh["ano"] = gh["year_month"].str.slice(0, 4).astype(int)
gh["mes"] = gh["year_month"].str.slice(5, 7).astype(int)

anos = sorted(gh["ano"].unique().tolist())
MESES = [("Janeiro", 1), ("Fevereiro", 2), ("Março", 3), ("Abril", 4),
         ("Maio", 5), ("Junho", 6), ("Julho", 7), ("Agosto", 8),
         ("Setembro", 9), ("Outubro", 10), ("Novembro", 11), ("Dezembro", 12)]
NOME_MES = {v: k for k, v in MESES}

def mostrar_humidade(ano, mes):
    d = gh[(gh["ano"] == ano) & (gh["mes"] == mes)].sort_values("district")
    if d.empty:
        print(f"Sem dados para {NOME_MES[mes]} {ano}.")
        return
    distritos = d["district"].tolist()
    x = np.arange(len(distritos)); w = 0.4
    plt.figure(figsize=(13, 6))
    plt.bar(x - w/2, d["humidade_max_mes"], width=w, label="humidade máx", color="#ff7f0e")
    plt.bar(x + w/2, d["humidade_min_mes"], width=w, label="humidade mín", color="#1f77b4")
    plt.ylabel("Humidade (%)")
    plt.xticks(x, distritos, rotation=45, ha="right")
    plt.title(f"Humidade máx/mín por distrito — {NOME_MES[mes]} {ano}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend(); plt.tight_layout(); plt.show()

interact(mostrar_humidade,
         ano=widgets.Dropdown(options=anos, description="Ano:"),
         mes=widgets.Dropdown(options=MESES, value=1, description="Mês:"));

interactive(children=(Dropdown(description='Ano:', options=(2025, 2026), value=2025), Dropdown(description='Mê…

In [59]:
gv = (spark.read.parquet(GOLD_MONTHLY)
        .select("district", "year_month", "vento_media_mes", "vento_max_mes", "rajada_max_mes")
        .toPandas())
gv["district"] = gv["district"].str.replace("%20", " ")
gv["ano"] = gv["year_month"].str.slice(0, 4).astype(int)
gv["mes"] = gv["year_month"].str.slice(5, 7).astype(int)

anos = sorted(gv["ano"].unique().tolist())
MESES = [("Janeiro", 1), ("Fevereiro", 2), ("Março", 3), ("Abril", 4),
         ("Maio", 5), ("Junho", 6), ("Julho", 7), ("Agosto", 8),
         ("Setembro", 9), ("Outubro", 10), ("Novembro", 11), ("Dezembro", 12)]
NOME_MES = {v: k for k, v in MESES}

def mostrar_vento(ano, mes):
    d = gv[(gv["ano"] == ano) & (gv["mes"] == mes)].sort_values("district")
    if d.empty:
        print(f"Sem dados para {NOME_MES[mes]} {ano}.")
        return
    distritos = d["district"].tolist()
    x = np.arange(len(distritos)); w = 0.27
    plt.figure(figsize=(14, 6))
    plt.bar(x - w, d["vento_media_mes"], width=w, label="vento médio", color="#1f77b4")
    plt.bar(x,     d["vento_max_mes"],   width=w, label="vento máx",   color="#ff7f0e")
    plt.bar(x + w, d["rajada_max_mes"],  width=w, label="rajada máx",  color="#d62728")
    plt.ylabel("Velocidade do vento (km/h)")
    plt.xticks(x, distritos, rotation=45, ha="right")
    plt.title(f"Vento por distrito — {NOME_MES[mes]} {ano}")
    plt.grid(True, axis="y", alpha=0.3)
    plt.legend(); plt.tight_layout(); plt.show()

interact(mostrar_vento,
         ano=widgets.Dropdown(options=anos, description="Ano:"),
         mes=widgets.Dropdown(options=MESES, value=1, description="Mês:"));

interactive(children=(Dropdown(description='Ano:', options=(2025, 2026), value=2025), Dropdown(description='Mê…

In [60]:
GOLD_DAILY = "hdfs://hadoop-master:39000/2250055/gold/gold_daily"

gd = (spark.read.parquet(GOLD_DAILY)
        .select("district", "date", "temp_max", "temp_min", "heat_index", "wind_chill")
        .toPandas())
gd["district"] = gd["district"].str.replace("%20", " ")
gd["date"] = pd.to_datetime(gd["date"])
gd["ano"] = gd["date"].dt.year
gd["mes"] = gd["date"].dt.month
gd["dia"] = gd["date"].dt.day

distritos = sorted(gd["district"].unique().tolist())
anos = sorted(gd["ano"].unique().tolist())
MESES = [("Janeiro", 1), ("Fevereiro", 2), ("Março", 3), ("Abril", 4),
         ("Maio", 5), ("Junho", 6), ("Julho", 7), ("Agosto", 8),
         ("Setembro", 9), ("Outubro", 10), ("Novembro", 11), ("Dezembro", 12)]
NOME_MES = {v: k for k, v in MESES}

def comparar(distrito, ano, mes):
    d = gd[(gd["district"] == distrito) & (gd["ano"] == ano) & (gd["mes"] == mes)].sort_values("dia")
    if d.empty:
        print(f"Sem dados para {distrito}, {NOME_MES[mes]} {ano}.")
        return
    plt.figure(figsize=(11, 6))
    plt.plot(d["dia"], d["temp_max"],   "-o", color="#ff7f0e", label="temp máx (real)")
    plt.plot(d["dia"], d["heat_index"], "--", color="#d62728", label="heat index (sensação de calor)")
    plt.plot(d["dia"], d["temp_min"],   "-o", color="#1f77b4", label="temp mín (real)")
    plt.plot(d["dia"], d["wind_chill"], "--", color="#17becf", label="wind chill (sensação de frio)")
    plt.xlabel("Dia do mês"); plt.ylabel("Temperatura (°C)")
    plt.title(f"{distrito} — {NOME_MES[mes]} {ano}: real vs sensação")
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

interact(comparar,
         distrito=widgets.Dropdown(options=distritos, description="Distrito:"),
         ano=widgets.Dropdown(options=anos, description="Ano:"),
         mes=widgets.Dropdown(options=MESES, value=1, description="Mês:"));

interactive(children=(Dropdown(description='Distrito:', options=('Aveiro', 'Beja', 'Braga', 'Braganca', 'Caste…

In [61]:
spark.stop()